### 소희 -상품분석서비스(점포_상권)

In [ ]:
from pathlib import Path
import pandas as pd
import zipfile
import shutil

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent
target_dir = project_root / 'data' / '01_영역'

# 연도별 파일명 (2024만 다름)
csv_names = {
    2019: '서울시_상권분석서비스(점포-상권)_2019년.csv',
    2020: '서울시_상권분석서비스(점포-상권)_2020년.csv',
    2021: '서울시_상권분석서비스(점포-상권)_2021년.csv',
    2022: '서울시_상권분석서비스(점포-상권)_2022년.csv',
    2023: '서울시_상권분석서비스(점포-상권)_2023년.csv',
    2024: '서울시 상권분석서비스(점포-상권)_2024년.csv',  # 공백!
}

dfs = {}

for year, csv_name in csv_names.items():
    csv_path = target_dir / csv_name
    
    if not csv_path.exists():
        print(f'📦 {year} CSV 없음, zip 찾는 중...')
        raw_dir = project_root / 'data' / 'raw'
        zip_files = list(raw_dir.glob(f'*점포*{year}*.zip'))
        
        if not zip_files:
            print(f'  ⚠️ {year} zip 없음, 건너뜀')
            continue
        
        zip_path = zip_files[0]
        target_dir.mkdir(parents=True, exist_ok=True)
        
        with zipfile.ZipFile(zip_path, 'r') as z:
            for info in z.infolist():
                if info.filename.startswith('__MACOSX'):
                    continue
                try:
                    filename = info.filename.encode('cp437').decode('utf-8')
                except:
                    filename = info.filename
                if info.is_dir():
                    continue
                out_path = target_dir / filename
                out_path.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as src, open(out_path, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    
    df = pd.read_csv(csv_path, encoding='cp949')
    dfs[year] = df
    print(f'✅ {year} 로드 완료: {df.shape}')

df_전체 = pd.concat(dfs.values(), ignore_index=True)
print(f'\n📊 전체 데이터: {df_전체.shape}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'AppleGothic'  # Mac 한글 폰트
plt.rcParams['axes.unicode_minus'] = False    # 마이너스 기호 깨짐 방지

* 전년도 불러오기 = df_전체
* 연도별 불러오기 = dfs[2019]

In [ ]:
# 연도/분기 기준으로 잘 집계되어있는지 확인.
display(dfs[2019]['기준_년분기_코드'].unique())
display(dfs[2020]['기준_년분기_코드'].unique())
display(dfs[2021]['기준_년분기_코드'].unique())
display(dfs[2022]['기준_년분기_코드'].unique())
display(dfs[2024]['기준_년분기_코드'].unique())

In [ ]:
# 점포수 null 값 확인
df_전체[['점포_수','유사_업종_점포_수','개업_율','폐업_률','폐업_점포_수','프랜차이즈_점포_수']].isnull().sum()

In [ ]:
#음수값 있는지 확인
(df_전체[['점포_수','유사_업종_점포_수','개업_율','폐업_률','폐업_점포_수','프랜차이즈_점포_수']]<0).sum()

In [ ]:
# 상품 구분코드가 어떤 명인지 확인.
display(dfs[2019][dfs[2019]['상권_구분_코드']=='A'][[ '상권_구분_코드', '상권_구분_코드_명']])
display(dfs[2019][dfs[2019]['상권_구분_코드']=='D'][[ '상권_구분_코드', '상권_구분_코드_명']])
display(dfs[2019][dfs[2019]['상권_구분_코드']=='R'][[ '상권_구분_코드', '상권_구분_코드_명']])
display(dfs[2019][dfs[2019]['상권_구분_코드']=='U'][[ '상권_구분_코드', '상권_구분_코드_명']])

In [ ]:
#2019~2024 null 값 없음.
df_전체.info()

# 상권 규모 분석

* 연도별 상권별 점포수 (트렌드)

In [ ]:

# 연도 컬럼 추가 (앞 4자리만 추출)
df_전체['연도'] = df_전체['기준_년분기_코드'].astype(str).str[:4].astype(int)

# 연도 + 상권구분별 평균 점포수
result= df_전체.groupby(['연도','상권_구분_코드_명'])['점포_수'].mean().reset_index()

# 피벗
pivot_store = result.pivot(index='상권_구분_코드_명', columns='연도', values='점포_수')
print(pivot_store)

In [ ]:
pivot_store.T.plot(kind='bar', figsize=(13, 6), edgecolor='white')

plt.title('상권 구분별 연도별 평균 점포 수', fontsize=15)
plt.xlabel('연도')
plt.ylabel('평균 점포 수')
plt.xticks(rotation=0)
plt.legend(title='상권 구분', loc='upper left')
plt.tight_layout()
plt.show()

# 상권별 폐업률 추이

In [ ]:

# 연도 컬럼 추가 (앞 4자리만 추출)
df_전체['연도'] = df_전체['기준_년분기_코드'].astype(str).str[:4].astype(int)

# 연도 + 평균 폐업률
연도별_폐업률= df_전체.groupby(['연도','상권_구분_코드_명'])['폐업_률'].mean().reset_index()

# 피벗
pivot_close = 연도별_폐업률.pivot(index='상권_구분_코드_명', columns='연도', values='폐업_률')
print(pivot_close.round(2))

In [ ]:
pivot_close.T.plot(kind='line', figsize=(10, 5), marker='o')
plt.title('상권별 폐업률 추이')
plt.ylabel('폐업률')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 상권별 개업률 추이

In [ ]:

# 연도 컬럼 추가 (앞 4자리만 추출)
df_전체['연도'] = df_전체['기준_년분기_코드'].astype(str).str[:4].astype(int)

# 연도 + 평균 폐업률
연도별_개업율= df_전체.groupby(['연도','상권_구분_코드_명'])['개업_율'].mean().reset_index()

# 피벗
pivot_open = 연도별_개업율.pivot(index='상권_구분_코드_명', columns='연도', values='개업_율')
print(pivot_open.round(2))

In [ ]:
pivot_open.T.plot(kind='line', figsize=(10, 5), marker='o')
plt.title('상권별 개업율 추이')
plt.ylabel('개업율')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 상권별 순성장률

* 상권유형별 순성장률

In [ ]:
#19~24년 순성장률
상권_구분_순성장률 = df_전체.groupby('상권_구분_코드_명').agg(
    평균개업률 = ('개업_율', 'mean'),
    평균폐업률 = ('폐업_률', 'mean')
).reset_index()

상권_구분_순성장률['순성장률'] = 상권_구분_순성장률['평균개업률'] - 상권_구분_순성장률['평균폐업률']
상권_구분_순성장률

In [ ]:
#연도별로 순성장률
상권_구분_순성장률 = df_전체.groupby(['연도','상권_구분_코드_명']).agg(
    평균개업률 = ('개업_율', 'mean'),
    평균폐업률 = ('폐업_률', 'mean')
).reset_index()

상권_구분_순성장률['순성장률'] = 상권_구분_순성장률['평균개업률'] - 상권_구분_순성장률['평균폐업률']


pivot = 상권_구분_순성장률.pivot(index='상권_구분_코드_명', columns='연도', values='순성장률')
print(pivot)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

for 상권 in pivot.index:
    ax.plot(pivot.columns, pivot.loc[상권], marker='o', label=상권)

ax.set_title('연도별 상권구분 순성장률', fontsize=14)
ax.set_xlabel('연도')
ax.set_ylabel('순성장률 (%)')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)  # 0 기준선
ax.legend(loc='best')
plt.tight_layout()
plt.show()

* 상권명별 순성장률

In [ ]:
상권_순성장률 = df_전체.groupby('상권_코드_명').agg(
    평균개업률 = ('개업_율', 'mean'),
    평균폐업률 = ('폐업_률', 'mean')
).reset_index()

상권_순성장률['순성장률'] = 상권_순성장률['평균개업률'] - 상권_순성장률['평균폐업률']
print(상권_순성장률)

In [ ]:
top_growth    = 상권_순성장률.nlargest(15, '순성장률').reset_index(drop=True)
bottom_growth = 상권_순성장률.nsmallest(15, '순성장률').reset_index(drop=True)

display(top_growth)
display(bottom_growth)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 성장 상권
top_sorted = top_growth.sort_values('순성장률', ascending=True)
axes[0].barh(top_sorted['상권_코드_명'], top_sorted['순성장률'], color='#4575b4')
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title('순성장률 상위 15개 상권', fontsize=13)
axes[0].set_xlabel('순성장률 (%)')

# Bottom 15 쇠퇴 상권
bottom_sorted = bottom_growth.sort_values('순성장률', ascending=False)
axes[1].barh(bottom_sorted['상권_코드_명'], bottom_sorted['순성장률'], color='#d73027')
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('순성장률 하위 15개 상권', fontsize=13)
axes[1].set_xlabel('순성장률 (%)')

plt.suptitle('상권별 순성장률 Top/Bottom 15', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()